# Simulating P vs. G

:::{sidebar} Discovering Game Theory
This course uses an **online textbook**: [Game Theory, a Discovery Approach](https://nordstrommath.com/IntroGameTheory2e/intro_game_theory.html), by Jennifer Nordstrom.
:::

Following on from the previous section, we would like to be able to construct a way to simulate the winning percentages of the various strategies. The simulation functions are described below.

In [13]:
## Do not change this cell, only execute it. 
## This cell initializes Python so that pandas, numpy and scipy packages are ready to use.

from datascience import *
import random
from IPython.display import display, Markdown
import pandas
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import numpy as np
import scipy as sp
import math
import string
import inflect
import scipy.stats as stats

engine = inflect.engine()

## Intial Values
We first initialize the values we need:
- $p$ : number of police units
- $g$ : number of guerrilla units
- strat : an arrays of strategies listed by larger number of the two groups of units
- index : the column or row numbers in the results table

In [14]:
p = 7
g = 5

### Dictionaries

Below, we create a function that creates player dictionaries to keep track of row-column indexes for the results table and strategies listed by integer value.

In [15]:
def create_player_dict(name, anchor_val):
    """
    Creates a dictionary containing player info, strategy sets, and index sets.
    """
    strat = np.arange(math.ceil(anchor_val / 2), anchor_val + 1)
    index = np.arange(1, len(strat) + 1)
    return {
        'player': name,
        'index': index,
        'strat': strat
    }

In [16]:
# Creating the dictionaries
P = create_player_dict('P', p)
G = create_player_dict('G', g)
G['index'] = G['index'] - 1

In [17]:
# Displaying the results
print(P)
print(G)

{'player': 'P', 'index': array([1, 2, 3, 4]), 'strat': array([4, 5, 6, 7])}
{'player': 'G', 'index': array([0, 1, 2]), 'strat': array([3, 4, 5])}


We can also display the indexes for a single player or the strategies for that player.

In [18]:
print(P['strat']) ; print(P['index']) ; print(G['strat']) ; print(G['index'])

[4 5 6 7]
[1 2 3 4]
[3 4 5]
[0 1 2]


### Words

The strategies we use can be indexed by integer, say, 7. However, the following function will covert integers to the appropriate words which, for the preceding case, would be "seven".

In [19]:
def words(num_array):

    return [engine.number_to_words(n) for n in num_array]

def word(n):
   return engine.number_to_words(n).replace("-", "_")

In [20]:
words(P['strat'])

['four', 'five', 'six', 'seven']

In [21]:
word(P['strat'][0])

'four'

### Init() Function
To initialize the results table, we wish to play each column strategy against each row strategy. We plan for 40 simulated battles with $10$ ghost results.  With $5$ ghost wins and $5$ ghost losses, the win percentages will skew to the center and away from the extremes of $0$ and $1$. The simulation results appear as though $50$ real results were simulated.

Adding ghost results is a traditional way to initialize proportions when we know some of those proporitions may be close to $0$ or $1$.

In [22]:
def init(p_max, g_max, p_strat, g_strat):
   A = p_strat ; B = p_max - A ;
   res = ()
   for k in range(40):
      a = np.random.choice(make_array( g_strat, g_max - g_strat ) ) ; b = g_strat - a ;
      if ( ( a > A ) or ( b > B ) ):
         v = 0
      else:
            v = 1
      res = np.append(res, v)
   temp = ( sum(res) + 5 ) / 50
   t = round(temp,3)
   return t

### Results Table

The following code simply sets up a table in which we plan to store the win percentages for each strategy matchup. The values in the table are irrelevant to the actual game and will serve as placeholders.

In [23]:
results = Table().with_columns('', words(G['strat']))
for t in P['index']:
    results = results.with_columns(word(P['strat'][t-1]),np.arange(len(G['strat'])).astype(np.float64)+t)
for t in P['index']:
  for s in G['index']:
    results[t][s] = init(p,g,P['strat'][t-1],G['strat'][s])

display(Markdown(f"<b>{p} Police vs. {g} Guerillas</b>"))
results

<b>7 Police vs. 5 Guerillas</b>

,four,five,six,seven
three,0.9,0.9,0.9,0.58
four,0.9,0.52,0.46,0.42
five,0.1,0.54,0.56,0.48


In [24]:
results[4][0]

0.57999999999999996

In [25]:
def play(p_0, g_0):
    A = np.random.choice(P['strat']) ; B = p_0 - A
    a = np.random.choice(G['strat']) ; b = g_0 - a
    if (a > A) or (b > B):
        v = 0
    else:
        v = 1
        
    return v, A, a  # This returns a "tuple" of three values

In [26]:
[val, pol, rilla] = play(7,5)
print(val)
print(pol)
print(rilla)

1
4
3


In [27]:
# for i in range(len(P['index'])):
#     print(P['index'][i])

In [28]:
#results = Table().with_columns('', words(G['strat']))
p_offset = P['strat'][1] - P['index'][1]
g_offset = G['strat'][1] - G['index'][1]
for t in P['index']:
   for s in range(len(G['index'])):
      results[t][s] = play(p,g,t+p_offset,s+g_offset)
results

TypeError: play() takes 2 positional arguments but 4 were given

In [59]:
def update_ema(current_ema, outcome, alpha=0.2):
    return (outcome * alpha) + (current_ema * (1 - alpha))

In [79]:
# results = Table().with_columns('', words(G['strat']))
for t in P['index']:
  for s in G['index']:
    results[t][s] = play(p,g,P['strat'][t-1],G['strat'][s]) 
results

,four,five,six,seven
three,0.9,0.9,0.9,0.44
four,0.9,0.56,0.58,0.42
five,0.1,0.48,0.54,0.5
